# Selección de puntos de pista

Marca los 10 puntos en este orden:

| # | Nombre | Descripción |
|---|--------|-------------|
| 1 | bottom_left | Esquina inferior-izquierda |
| 2 | bottom_right | Esquina inferior-derecha |
| 3 | top_right | Esquina superior-derecha |
| 4 | top_left | Esquina superior-izquierda |
| 5 | service_near_left | Línea de saque lado bottom, izquierda |
| 6 | service_near_right | Línea de saque lado bottom, derecha |
| 7 | service_far_left | Línea de saque lado top, izquierda |
| 8 | service_far_right | Línea de saque lado top, derecha |
| 9 | T_near | T central lado bottom |
| 10 | T_far | T central lado top |

In [4]:
GAME = "game"   # <- cambia aquí el game

In [5]:
!pip install ipycanvas

In [7]:
import io, json, sys
from pathlib import Path
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from ipycanvas import Canvas

sys.path.insert(0, str(Path(".").resolve()))
import config
from src.geometry.homography import POINT_ORDER, WORLD_POINTS

N_POINTS = len(POINT_ORDER)
CLICK_LABELS = [
    "1 inf-izq", "2 inf-der", "3 sup-der", "4 sup-izq",
    "5 saque↓izq", "6 saque↓der",
    "7 saque↑izq", "8 saque↑der",
    "9 T↓", "10 T↑",
]
COLORS = [
    "red", "blue", "lime", "orange",
    "violet", "cyan", "darkorange", "gray", "deeppink", "deepskyblue",
]

# Cargar frame de referencia
frame_path = config.DATASET_ROOT / GAME / "Clip1" / config.REFERENCE_FRAME
if not frame_path.exists():
    for candidate in (config.OUTPUTS_ROOT / GAME).rglob(config.REFERENCE_FRAME):
        frame_path = candidate
        break
assert frame_path.exists(), f"No se encontró el frame: {frame_path}"

img_bgr = cv2.imread(str(frame_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H_px, W_px = img_rgb.shape[:2]

# Escalar para que quepa en pantalla
DISPLAY_W = 960
scale = DISPLAY_W / W_px
DISPLAY_H = int(H_px * scale)

# Array RGBA para put_image_data
img_resized = cv2.resize(img_rgb, (DISPLAY_W, DISPLAY_H))

img_rgba = np.dstack([img_resized, np.full((DISPLAY_H, DISPLAY_W), 255, dtype=np.uint8)])

out_dir    = config.OUTPUTS_ROOT / GAME
out_dir.mkdir(parents=True, exist_ok=True)
court_file = out_dir / config.COURT_POINTS_FILENAME

print(f"Frame : {frame_path}  ({W_px}x{H_px})")
print(f"Salida: {court_file}")
if court_file.exists():
    with open(court_file) as f:
        _d = json.load(f)
    print(f"Ya existe con {len(_d['image_points'])} puntos (se sobreescribirá al guardar)")

Frame : /workspace/TFM/Dataset/game/Clip1/0000.jpg  (1280x720)
Salida: /workspace/TFM/outputs/game/court_points.json


In [8]:
# ── Selector interactivo ───────────────────────────────────────────────────
selected_pts = []   # coordenadas en píxeles ORIGINALES (no escalados)

canvas = Canvas(width=DISPLAY_W, height=DISPLAY_H)
status = widgets.Label(value=f"Clic en el punto 1: {POINT_ORDER[0]}  [{CLICK_LABELS[0]}]")
btn_undo = widgets.Button(description="↩ Deshacer", button_style="warning",
                          layout=widgets.Layout(width="130px"))
btn_save = widgets.Button(description="Guardar ✓", button_style="success",
                          disabled=True, layout=widgets.Layout(width="130px"))

def _redraw():
    canvas.put_image_data(img_rgba, 0, 0)
    for i, pt in enumerate(selected_pts):
        sx, sy = pt[0] * scale, pt[1] * scale
        canvas.fill_style = COLORS[i]
        canvas.fill_arc(sx, sy, 7, 0, 6.2832)
        canvas.stroke_style = "white"
        canvas.line_width = 1.5
        canvas.stroke_arc(sx, sy, 7, 0, 6.2832)
        canvas.fill_style = "white"
        canvas.font = "bold 11px sans-serif"
        canvas.fill_text(CLICK_LABELS[i], sx + 9, sy - 5)

def _update_status():
    n = len(selected_pts)
    if n < N_POINTS:
        status.value = f"Clic en el punto {n+1}: {POINT_ORDER[n]}  [{CLICK_LABELS[n]}]"
        btn_save.disabled = True
    else:
        status.value = f"✓ {N_POINTS} puntos listos. Pulsa Guardar."
        btn_save.disabled = False

def _on_mouse_down(x, y):
    if len(selected_pts) >= N_POINTS:
        return
    selected_pts.append([x / scale, y / scale])
    _redraw()
    _update_status()

def _on_undo(_):
    if selected_pts:
        selected_pts.pop()
        _redraw()
        _update_status()

def _on_save(_):
    if len(selected_pts) != N_POINTS:
        status.value = f"Faltan puntos: {len(selected_pts)}/{N_POINTS}"
        return
    payload = {
        "cache_key": GAME,
        "reference_frame": str(frame_path),
        "frame_size": [W_px, H_px],
        "court_type": config.COURT_TYPE,
        "image_points": selected_pts,
        "world_points": WORLD_POINTS.tolist(),
        "point_order": POINT_ORDER,
    }
    with open(court_file, "w") as f:
        json.dump(payload, f, indent=2)
    status.value = f"✓ Guardado en {court_file}"
    btn_save.disabled = True
    btn_undo.disabled = True
    print(f"Guardado: {court_file}")
    for i, (name, pt) in enumerate(zip(POINT_ORDER, selected_pts), 1):
        print(f"  P{i:2d} {name:25s}: ({pt[0]:.1f}, {pt[1]:.1f})")

canvas.on_mouse_down(_on_mouse_down)
btn_undo.on_click(_on_undo)
btn_save.on_click(_on_save)

_redraw()
display(widgets.VBox([
    status,
    widgets.HBox([btn_undo, btn_save]),
    canvas,
]))

In [ ]:
# ── Verificación: error de reproyección ────────────────────────────────────
from src.geometry.homography import compute_homography, project_points

with open(court_file) as f:
    d = json.load(f)

img_pts   = np.array(d["image_points"], dtype=np.float32)
H         = compute_homography(img_pts)
recovered = project_points(img_pts, H)
world     = WORLD_POINTS[:len(img_pts)]
errs      = np.linalg.norm(recovered - world, axis=1)

print(f"Puntos usados : {len(img_pts)}")
print(f"Error medio   : {errs.mean():.4f} m")
print(f"Error máximo  : {errs.max():.4f} m")
print()
for i, (name, err) in enumerate(zip(POINT_ORDER[:len(img_pts)], errs), 1):
    flag = "  ← alto" if err > 0.3 else ""
    print(f"  P{i:2d} {name:25s}: {err:.4f} m{flag}")

Puntos usados : 10
Error medio   : 9.5961 m
Error máximo  : 24.2047 m

  P 1 bottom_left              : 23.7403 m  ←← alto
  P 2 bottom_right             : 24.0572 m  ←← alto
  P 3 top_right                : 24.2047 m  ←← alto
  P 4 top_left                 : 23.6527 m  ←← alto
  P 5 service_near_left        : 0.0454 m
  P 6 service_near_right       : 0.0290 m
  P 7 service_far_left         : 0.0555 m
  P 8 service_far_right        : 0.0295 m
  P 9 T_near                   : 0.0662 m
  P10 T_far                    : 0.0806 m
